# Day 2 — Tools and Memory

**An LLM on its own can talk. An agent with tools can *do things*.**

Yesterday you built a chat loop and met the OpenAI Agents SDK. Today you'll learn how to give that agent **tools** (so it can search the web, read files, fetch live data), how to give it **memory** (so it remembers your conversation), and how to make the resulting code **production-ready** (so it doesn't fall apart the moment something goes wrong).

By the end of the morning you'll have all the pieces to build the kind of agent you'd actually want to use: one that searches, fetches, remembers, and behaves sensibly.

## Learning outcomes

By 13:00 you should be able to:

1. Design a well-shaped tool and wire it into an agent.
2. Use **structured outputs** to make agent responses predictable.
3. Use **sessions** to give the agent multi-turn memory.
4. Test your tools with **pytest** so you'd trust them in production.
5. Recognise the patterns that separate notebook code from shippable code.

## How to use this notebook

Same conventions as Day 1. Labels tell you what's core lesson, what's reference, and what you should try yourself:

| Label | Meaning |
|---|---|
| 🟢 **Core lesson** | Follow during the morning session |
| 🔍 **Predict before running** | Pause, decide what you expect, then run |
| ✅ **Check your understanding** | A short exercise to verify the concept |
| 📚 **Self-study** | Read on your own time — covered in the notebook, not in class |
| ⚠️ **Production note** | Important engineering context |

## Three parts

| Part | Topic |
|---|---|
| **Part 1** | Agents with Tools |
| **Part 2** | Memory in Agent Systems |
| **Part 3** | Making it production-ready |

The notebook is also a *toolkit*. Many tool examples sit in the notebook for you to copy, adapt, and use in this afternoon's lab — even if we don't walk through every one in class.

---

## Setup

Same shared course environment as Day 1. The cell below verifies the environment and fails fast if anything is missing.

In [1]:
import os
import sys
from importlib.metadata import PackageNotFoundError, version
from dotenv import load_dotenv

if sys.version_info[:2] != (3, 13):
    raise RuntimeError(
        f"This course uses Python 3.13. Current interpreter: {sys.version.split()[0]}"
    )

load_dotenv()

required = ["OPENAI_API_KEY", "TAVILY_API_KEY"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing environment variables: {', '.join(missing)}. "
        "Check the course .env file."
    )

print(f"Python: {sys.version.split()[0]}")
print("Environment variables loaded.")

Python: 3.13.13
Environment variables loaded.


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from agents import Agent, Runner, SQLiteSession, function_tool

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"Using model: {MODEL}")

for pkg in ("openai-agents",):
    try:
        print(f"{pkg}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg}: not installed")

Using model: gpt-4o-mini
openai-agents: 0.17.6


---

# Part 1 — Agents with Tools

An agent without tools is just a chatbot. It can talk, reason, and produce text — but it cannot *do* anything outside the conversation. It can't check the weather, look up a price, search the web, or read a file.

**Tools change that.** A tool is a function the agent can call, with arguments the model chooses based on the conversation. The model decides *when* to call a tool, *what* arguments to pass, and *what to do with the result*.

This part covers:

1. What a tool actually is, mechanically
2. Designing tools with clear inputs and outputs
3. Structured outputs from the agent itself
4. Real-world tools — file I/O, external APIs, web search
5. Prompt design — telling the agent how to use its tools well

## 1.1 What is a tool?

🟢 **Core lesson**

A tool is just a Python function with three properties the model needs:

1. **A name** — what the tool is called
2. **A description** — what the tool does, written for the model to read
3. **A signature** — the inputs it accepts and the output it returns

The OpenAI Agents SDK wraps this with the `@function_tool` decorator. It introspects your function's signature, generates a JSON schema, and exposes it to the model.

```mermaid
flowchart LR
    U[User message] --> A[Agent]
    A -->|decides to call tool| T[Tool function]
    T -->|returns result| A
    A -->|uses result| R[Agent response]
    A --> R
```

The agent loops on this: read input, decide whether to call a tool, call it, read the result, decide again. It stops when it has a final answer.

## 1.2 Your first tool — `get_weather`

🟢 **Core lesson**

Let's start with the simplest possible tool: one that returns fake weather data. No API, no errors to handle, no JSON to parse — just a function that returns a string. This lets us focus on the **mechanics** of how a tool gets wired up.

> **Why fake?** Real APIs add noise — keys, rate limits, error handling. We want to learn the *pattern* first. We'll move to real APIs in Section 1.5.

In [3]:
@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: the name of the city to check, e.g. "Almaty" or "London".

    Returns:
        A short natural-language weather summary.
    """
    # A stubbed implementation — pretends to look up the weather.
    fake_data = {
        "almaty": "Cold and clear, -3°C",
        "london": "Overcast with light rain, 8°C",
        "astana": "Heavy snow, -12°C",
    }
    return fake_data.get(city.lower(), f"Sorry, no data for {city}.")


print("Tool defined.")
print(f"Tool name: {get_weather.name}")
print(f"Tool description: {get_weather.description}")

Tool defined.
Tool name: get_weather
Tool description: Get the current weather for a city.


In [4]:
get_weather.params_json_schema

{'properties': {'city': {'description': 'the name of the city to check, e.g. "Almaty" or "London".',
   'title': 'City',
   'type': 'string'}},
 'required': ['city'],
 'title': 'get_weather_args',
 'type': 'object',
 'additionalProperties': False}

**Three things to notice:**

1. **The function is small.** Five lines of real logic. Tools should be *focused* — one job each.
2. **The docstring matters.** The SDK reads it and uses it as the tool's description. The model sees this. Write it for the model.
3. **Type hints matter.** `city: str` tells the model the input is a string. The SDK turns this into a JSON schema the model can reason about.

Let's wire it into an agent.

In [7]:
weather_agent = Agent(
    name="Weather Assistant",
    instructions="You are a friendly weather assistant. Use the get_weather tool to answer questions about the weather.",
    tools=[get_weather],
    model=MODEL,
)

result = await Runner.run(weather_agent, "What's the weather like in Almaty today?")
print(result.final_output)

The weather in Almaty today is cold and clear, with a temperature of -3°C.


In [8]:
# Case where the tool does not know the city
result = await Runner.run(weather_agent, "What's the weather like in Tokyo today?")
print(result.final_output)

It seems I can't retrieve the weather data for Tokyo at the moment. If you're able to check another city or need help with something else, just let me know!


🔍 **Predict before running:** what happens if you ask about a city not in `fake_data`?

✅ **Check your understanding:** add a second tool `get_humidity(city)` that returns a stub humidity percentage. Give the agent both tools. Ask it a question that needs both.

## 1.3 Inputs done well

🟢 **Core lesson**

A tool with vague inputs is a tool the model will call wrong. Three habits make tool inputs robust:

1. **Type everything.** `str`, `int`, `float`, `bool`, `list[str]` — the model reads these.
2. **Constrain when you can.** Use `Literal[...]` for known options. The model won't try to invent values.
3. **Use Pydantic for structured inputs.** When a tool needs a group of related fields, model them as a Pydantic class.

Let's see all three in action.

In [10]:
@function_tool
def search_books(
    query: str,
    max_results: int = 5,
    sort_by: Literal["relevance", "date", "rating"] = "relevance",
) -> str:
    """Search for books matching a query.

    Args:
        query: what to search for (e.g. "agentic AI").
        max_results: how many results to return (1-20).
        sort_by: how to order results — by relevance, date, or rating.

    Returns:
        A short formatted list of matching books.
    """
    # Stubbed — pretends to call a book API.
    return (
        f"Top {max_results} books matching '{query}', sorted by {sort_by}:\n"
        f"1. The Agentic Future (2025)\n"
        f"2. Tools and Memory (2024)\n"
        f"3. Production AI Engineering (2024)"
    )


print("Tool with constrained inputs ready.")

Tool with constrained inputs ready.


**Why `Literal[...]` matters:** without it, the model might pass `sort_by="random"` or `sort_by="newest"` — values your function doesn't handle.  
With `Literal["relevance", "date", "rating"]`, the model knows the only valid choices.

For more complex inputs, use a Pydantic model:

In [11]:
class Address(BaseModel):
    street: str
    city: str
    postcode: str
    country: str = "United Kingdom"


@function_tool
def estimate_delivery(address: Address, weight_kg: float) -> str:
    """Estimate delivery time and cost for a package.

    Args:
        address: the destination address.
        weight_kg: package weight in kilograms.

    Returns:
        A delivery estimate.
    """
    # Stubbed.
    return f"Estimated delivery to {address.city}: 3 days, £{weight_kg * 2.5:.2f}"


print("Tool with Pydantic input ready.")

Tool with Pydantic input ready.


⚠️ **Production note — schema-compatible types only.**

The model can only pass JSON-compatible values. That means: `str`, `int`, `float`, `bool`, `list[...]`, `dict[str, ...]`, `Literal[...]`, and Pydantic models.  
**`Path`, `datetime`, `UUID` will fail** — the OpenAI function-calling schema doesn't recognise them. If a tool needs one of these internally, declare it as `str` and parse inside the function.

## 1.4 Structured outputs from the agent

🟢 **Core lesson**

So far our agents have returned free-form text. That's fine for chat, but it's terrible for code that needs to *do* something with the response.

If you want to display a list of books in a UI, write rows to a spreadsheet, or pass the agent's output to another system, you don't want a paragraph — you want **structured data**.

The `output_type=` parameter tells the agent to return a Pydantic model instead of text.

In [12]:
class BookRecommendation(BaseModel):
    title: str
    author: str
    year: int
    why_recommended: str = Field(description="A one-sentence reason this book matches the user's request.")


class BookRecommendations(BaseModel):
    user_interest: str = Field(description="A short summary of what the user is interested in.")
    recommendations: list[BookRecommendation]


book_agent = Agent(
    name="Book Recommender",
    instructions=(
        "You recommend books based on what the user is interested in. "
        "Always return three recommendations with a clear reason for each."
    ),
    output_type=BookRecommendations,
    model=MODEL,
)

result = await Runner.run(
    book_agent,
    "I want to learn about how language models work under the hood.",
)

# result.final_output is now a BookRecommendations instance, not a string.
output = result.final_output
print(f"User interest: {output.user_interest}\n")
for book in output.recommendations:
    print(f"📖 {book.title} by {book.author} ({book.year})")
    print(f"   {book.why_recommended}\n")

User interest: Learning about the mechanics and principles behind language models.

📖 Deep Learning for Natural Language Processing by Palash Goyal (2021)
   This book offers a comprehensive introduction to deep learning techniques used in NLP, explaining concepts with clarity.

📖 Speech and Language Processing by Daniel Jurafsky and James H. Martin (2020)
   A widely-used textbook that covers the fundamentals of NLP, including detailed insights into language models and their functionalities.

📖 Transformers for Natural Language Processing by Denis Rothman (2020)
   This book focuses on transformer models, which are central to modern language processing, with practical examples and applications.



**That's the magic.** The agent's response is now a Python object with typed fields. You can pass it to other code, store it in a database, render it as JSON — anything you'd do with normal Python data.

This pairs beautifully with tools. The model can call tools to *gather* information, then return its findings as a structured object.

## 1.5 Tools that call the world

🟢 **Core lesson** *(one example walked through live)*

Time to leave stub-land. Real tools call external APIs, read files, or interact with the local environment.

The mechanics are the same — `@function_tool`, a docstring, typed inputs. The only difference is that *real things can fail*. Networks time out, files don't exist, APIs return errors. **A good tool returns a clear message when this happens**, instead of raising an exception that confuses the agent.

We'll walk through one real example live: **crypto prices from CoinGecko**. The other tools below are for you to copy, adapt, and use in the lab.

### Live walkthrough — `get_crypto_price`

Uses the [CoinGecko public API](https://www.coingecko.com/en/api). No API key needed for simple queries.

In [13]:
import httpx

#@function_tool
def _get_crypto_price(symbol: str, currency: str = "usd") -> str:
    """Get the current price of a cryptocurrency.

    Args:
        symbol: the coin id used by CoinGecko (e.g. 'bitcoin', 'ethereum', 'solana').
        currency: the target currency (default 'usd').

    Returns:
        A short natural-language price summary, or an error message if the call failed.
    """
    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {"ids": symbol, "vs_currencies": currency}

    try:
        response = httpx.get(url, params=params, timeout=10.0)
        response.raise_for_status()
        data = response.json()
    except httpx.HTTPError as exc:
        return f"Error: could not reach CoinGecko. Reason: {exc}"

    if symbol not in data:
        return f"Error: '{symbol}' not found. Try 'bitcoin', 'ethereum', or 'solana'."

    price = data[symbol].get(currency)
    if price is None:
        return f"Error: no {currency.upper()} price available for {symbol}."

    return f"{symbol.capitalize()} is currently {price} {currency.upper()}."


# Wrap it as a tool for the agent.
get_crypto_price = function_tool(_get_crypto_price)


# Test the underscore version directly — no agent involved.
print(_get_crypto_price("bitcoin", "usd"))

Bitcoin is currently 63168 USD.


**Notice the error handling.** Three different failure modes — network error, unknown symbol, missing currency — each returns a clear string. The agent reads these like any other tool result and can react sensibly (*"That coin doesn't seem to exist — did you mean bitcoin?"*).

Now wire it into an agent:

In [14]:
crypto_agent = Agent(
    name="Crypto Helper",
    instructions=(
        "You help users check cryptocurrency prices. "
        "Use the get_crypto_price tool whenever the user asks about a coin's price. "
        "If the tool returns an error, explain it gently."
    ),
    tools=[get_crypto_price],
    model=MODEL,
)

result = await Runner.run(crypto_agent, "What's the price of bitcoin in GBP?")
print(result.final_output)

The current price of Bitcoin is £47,763.


---

### 📚 Self-study tools

The tools below follow the same pattern. We won't walk through them live — but they're ready to use in this afternoon's lab. Copy what you need.

#### Web search with Tavily

In [15]:
#@function_tool
def _search_web(query: str, max_results: int = 3) -> str:
    """Search the web using Tavily and return a short summary of results.

    Args:
        query: a clear search query.
        max_results: how many results to return (default 3).

    Returns:
        A formatted summary of the top results.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return "Error: TAVILY_API_KEY is not set."

    try:
        response = httpx.post(
            "https://api.tavily.com/search",
            json={"api_key": api_key, "query": query, "max_results": max_results},
            timeout=15.0,
        )
        response.raise_for_status()
        data = response.json()
    except httpx.HTTPError as exc:
        return f"Error: search failed. Reason: {exc}"

    results = data.get("results", [])
    if not results:
        return f"No results for: {query}"

    lines = []
    for r in results:
        title = r.get("title", "(no title)")
        url = r.get("url", "(no url)")
        snippet = r.get("content", "")[:200]
        lines.append(f"- {title}\n  {url}\n  {snippet}...")

    return f"Top {len(results)} results for '{query}':\n\n" + "\n\n".join(lines)


# Wrap it as a tool for the agent.
search_web = function_tool(_search_web)


print("search_web defined.")

search_web defined.


#### Gold price (from [Gold API](https://gold-api.com/) — free, no key needed)

In [55]:
#@function_tool
def _get_gold_price_usd_per_troy_ounce() -> str:
    """
    Get the latest gold price in USD per troy ounce.

    Returns:
        A human-readable result string.
        Example: "Gold price: $2,350.25 USD per troy ounce"

    Notes:
        XAU usually represents 1 troy ounce of gold.

        1 troy ounce = 31.1034768 grams.
        1 kilogram = 32.1507466 troy ounces.

        Uses Gold API.
    """
    try:
        response = httpx.get(
            "https://api.gold-api.com/price/XAU",
            timeout=10,
        )
        response.raise_for_status()

        data: dict[str, Any] = response.json()
        price = data["price"]

        return f"Gold price: ${float(price):,.2f} USD per troy ounce"

    except httpx.HTTPError as exc:
        return f"Error: could not fetch the gold price. Reason: {exc}"
    except (KeyError, TypeError, ValueError) as exc:
        return f"Error: gold price service returned unexpected data. Reason: {exc}"
    

# Wrap it as a tool for the agent.
get_gold_price_usd_per_troy_ounce = function_tool(_get_gold_price_usd_per_troy_ounce)

print(_get_gold_price_usd_per_troy_ounce())

Gold price: $4,156.70 USD per troy ounce


#### Currecny Rate (from Frankfurter — free, no key needed)

In [54]:
#@function_tool
def _get_currency_rate(base: str, target: str) -> str:
    """
    Get the latest exchange rate between two fiat currencies.

    Args:
        base: Base currency code, such as "GBP".
        target: Target currency code, such as "USD".

    Returns:
        A human-readable result string.
        Example: "1 GBP = 1.2700 USD"
    """
    base = base.upper().strip()
    target = target.upper().strip()

    if not base or not target:
        return "Error: base and target currency codes are required."

    try:
        response = httpx.get(
            f"https://api.frankfurter.dev/v2/rate/{base}/{target}",
            timeout=10,
        )
        response.raise_for_status()

        data: dict[str, Any] = response.json()
        rate = float(data["rate"])

        return f"1 {base} = {rate:,.4f} {target}"

    except httpx.HTTPError as exc:
        return (
            f"Error: could not fetch the currency rate for {base} to {target}. "
            f"Reason: {exc}"
        )
    except (KeyError, TypeError, ValueError) as exc:
        return (
            f"Error: currency service returned unexpected data for {base} to {target}. "
            f"Reason: {exc}"
        )


# Wrap it as a tool for the agent.
get_currency_rate = function_tool(_get_currency_rate)

    
print(_get_currency_rate("GBP", "USD"))

1 GBP = 1.3253 USD


#### File I/O — read and write text files

In [17]:
from pathlib import Path

OUTPUTS_DIR = Path("outputs")
OUTPUTS_DIR.mkdir(exist_ok=True)


#@function_tool
def _write_text_file(filename: str, content: str) -> str:
    """Save text content to a file under outputs/.

    Args:
        filename: file name (no path separators allowed).
        content: text to write.

    Returns:
        A confirmation message with the saved path.
    """
    if "/" in filename or "\\" in filename or ".." in filename:
        return f"Error: filename must not contain path separators. Got: {filename!r}"

    path = OUTPUTS_DIR / filename
    try:
        path.write_text(content, encoding="utf-8")
    except OSError as exc:
        return f"Error: could not write {filename}. Reason: {exc}"

    return f"Saved to {path}"

# Wrap it as a tool for the agent.
write_text_file = function_tool(_write_text_file)


#@function_tool
def _read_text_file(filename: str) -> str:
    """Read a text file from outputs/.

    Args:
        filename: file name (no path separators allowed).

    Returns:
        The file contents, or an error message.
    """
    if "/" in filename or "\\" in filename or ".." in filename:
        return f"Error: filename must not contain path separators. Got: {filename!r}"

    path = OUTPUTS_DIR / filename
    if not path.exists():
        return f"Error: file {filename} not found in outputs/."

    return path.read_text(encoding="utf-8")


# Wrap it as a tool for the agent.
read_text_file = function_tool(_read_text_file)

print("File I/O tools defined.")

File I/O tools defined.


⚠️ **Production note — path traversal.** Every file tool above rejects filenames containing `/`, `\\`, or `..`. This isn't a hard security boundary, but it stops the obvious failure mode where a clever prompt convinces the agent to write to `/etc/passwd` or similar.

## 1.6 Prompts and instruction (system prompt) templates

🟢 **Core lesson**

You've seen the `instructions=` parameter on every agent. It's the prompt — the *constant guidance* the model sees on every turn.

Good agent instructions are:

1. **Specific about the role.** Not "be helpful" but "you are a friendly crypto-price assistant."
2. **Explicit about tools.** Tell the model *when* to use each tool, not just that it exists.
3. **Clear about output.** If the agent should be concise, say so. If it should refuse certain requests, say so.

Bad instructions produce inconsistent agents. Good instructions produce predictable ones.

In [18]:
# A weak prompt
weak_agent = Agent(
    name="Weak Agent",
    instructions="You are helpful.",
    tools=[get_crypto_price, get_weather],
    model=MODEL,
)

# A stronger prompt
strong_agent = Agent(
    name="Strong Agent",
    instructions=(
        "You are a personal briefing assistant for crypto and weather.\n\n"
        "When the user asks about a cryptocurrency, use get_crypto_price. "
        "When they ask about the weather, use get_weather. "
        "If they ask for both in one message, call both tools.\n\n"
        "Keep your answers to one or two sentences. "
        "If a tool returns an error, explain it briefly and suggest an alternative."
    ),
    tools=[get_crypto_price, get_weather],
    model=MODEL,
)

print("Two versions of the same agent, with different instructions.")

Two versions of the same agent, with different instructions.


### Templates for repeatable prompts

When you're building several similar agents — or running the same agent against many inputs — store the prompt as a template instead of hard-coding strings.

A simple pattern:

In [19]:
BRIEFING_AGENT_INSTRUCTIONS = """
You are a {role}.

When the user asks about {topic_a}, use {tool_a}.
When the user asks about {topic_b}, use {tool_b}.

Keep answers to {max_sentences} sentences.
If a tool returns an error, {error_handling}.
""".strip()


# Build an agent from the template.
crypto_briefer = Agent(
    name="Crypto Briefer",
    instructions=BRIEFING_AGENT_INSTRUCTIONS.format(
        role="personal crypto briefing assistant",
        topic_a="cryptocurrency prices",
        tool_a="get_crypto_price",
        topic_b="weather",
        tool_b="get_weather",
        max_sentences="two",
        error_handling="explain the error and suggest an alternative",
    ),
    tools=[get_crypto_price, get_weather],
    model=MODEL,
)

print("Templated agent ready.")

Templated agent ready.


**Why this matters in production:** when prompts live in templates, they can be versioned, A/B tested, and improved without touching agent code. We'll come back to this idea in Part 3.

---

# Part 2 — Memory in Agent Systems

You've built agents that call tools. They work — but they have a critical limitation: **they forget everything between runs.**

```python
await Runner.run(agent, "My name is Aida.")
await Runner.run(agent, "What's my name?")   # Agent has no idea.
```

Each `Runner.run()` is an independent call. The model sees only that turn's input — no history, no context, no memory of what was said before.

**This part is about adding memory.** We'll cover the conceptual landscape briefly, then implement what you'll actually use today: **sessions** for multi-turn conversations.

## 2.1 The mental model — what memory means for agents

🟢 **Core lesson**

The word *memory* gets used loosely. Two distinctions are worth getting straight before any code:

### LLM memory vs Agent memory

There are two completely different things called "memory":

- **LLM memory** is what the model has access to during one call — the context window. Static, transient, gone when the call ends.
- **Agent memory** is everything *you build on top of the LLM* to give the agent continuity across calls. It's a storage layer outside the model.

The LLM is the cognitive engine. **Agent memory is the storage you wrap around it.**

```mermaid
flowchart TB
    A[Agent] --> M[Memory storage]
    M --> A
    A --> L[LLM with context window]
    L --> A
    style M fill:#4a5568,color:#fff
```

## 2.2 Two kinds — short-term and long-term

🟢 **Core lesson**

Agent memory splits into two categories, separated by **scope and lifetime**:

| | Short-term memory | Long-term memory |
|---|---|---|
| **Purpose** | Continue the current conversation | Remember across conversations |
| **Scope** | One session or thread | A user, across many sessions |
| **Contents** | Recent messages, tool calls | Preferences, facts, decisions |
| **Lifetime** | The session lasts | Until explicitly deleted |
| **Example** | *"Make the meeting 15:00 instead"* | *"Aida prefers concise answers"* |

Both categories show up everywhere — the [CoALA paper](https://arxiv.org/abs/2309.02427), [IBM's primer](https://www.ibm.com/think/topics/ai-agent-memory), [LangChain's memory guide](https://docs.langchain.com/oss/python/concepts/memory), [Mem0's docs](https://docs.mem0.ai/core-concepts/memory-types). The vocabulary is settled.

**Today we focus on short-term memory** — it's what makes multi-turn conversation work, and it's enough for the kind of agent you'll build in the lab. The next section gives the short version of what long-term memory looks like in practice.

### What long-term memory looks like in practice

📚 **For context — no implementation today**

A long-term memory layer remembers things the user said in *previous* conversations. It's used for personalisation: *"Aida prefers worked examples"*, *"Boris is a vegetarian"*, *"this user already covered transformer basics last week"*.

How it works mechanically:

1. **Extract** — decide what's worth remembering from a conversation
2. **Embed** *(optional)* — convert each fact to a vector for semantic search
3. **Store** — write to a database
4. **Retrieve** — query the database when a new conversation starts

You can build this yourself on top of any database. There are also managed services that do all four steps for you. The most common:

| Service | What it does |
|---|---|
| **[Mem0](https://mem0.ai)** | Managed memory with extraction + embedding + retrieval |
| **[Zep](https://www.getzep.com/)** | Temporal knowledge graphs — tracks how facts evolve over time |
| **[Letta](https://github.com/letta-ai/letta)** *(formerly MemGPT)* | The agent manages its own memory tiers |

You don't need any of these to build something cool today. If you want to dig deeper, all three have free tiers and good docs.

## 2.3 Sessions in practice

🟢 **Core lesson**


A [`Session`](https://openai.github.io/openai-agents-python/sessions/) is the primary abstraction for short-term conversational memory in the OpenAI Agents SDK. It acts as an stateful object that automatically appends user inputs, assistant responses, and tool traces during execution.  

When injected into `Runner.run()`, the SDK natively handles retrieving past context before the model runs and persisting the updated log immediately after, supporting backends like SQLite, Redis, and MongoDB.




```mermaid
sequenceDiagram
    participant U as User
    participant R as Runner
    participant S as Session
    participant M as Model

    U->>R: New message
    R->>S: Load previous items
    S-->>R: Conversation history
    R->>M: History + new message
    M-->>R: Response
    R->>S: Store new items
    R-->>U: Final response
```

You never call `.to_input_list()` or manage conversation state yourself.

🔍 **Predict before running:** the two runs below share a session. Will the second run know the name from the first?

Additonal Reading Material: https://developers.openai.com/cookbook/examples/agents_sdk/session_memory

In [20]:
# Without a session — the agent forgets.

stateless = Agent(
    name="Stateless",
    instructions="You are a concise assistant.",
    model=MODEL,
)

a = await Runner.run(stateless, "Hi, I'm Aida.")
b = await Runner.run(stateless, "What's my name?")

print(f"Run 1: {a.final_output}")
print(f"Run 2: {b.final_output}")

Run 1: Hi Aida! How can I assist you today?
Run 2: I don’t know your name. How can I assist you today?


In [21]:
# With a session — the agent remembers.

stateful = Agent(
    name="Stateful",
    instructions="You are a concise assistant. Use any prior conversation history.",
    model=MODEL,
)

session = SQLiteSession("aida_demo")

a = await Runner.run(stateful, "Hi, I'm Aida.", session=session)
b = await Runner.run(stateful, "What's my name?", session=session)

print(f"Run 1: {a.final_output}")
print(f"Run 2: {b.final_output}")

Run 1: Hi Aida! How can I assist you today?
Run 2: Your name is Aida.


**The agent didn't change.** The application supplied earlier session items to the model on the second run. That's all *adding memory* means at this level.

### Inspect the Session Object


You can inspect what's stored in a session at any time using `get_items()`.  
This is useful for debugging — when an agent behaves unexpectedly, look at what it actually saw.



In [22]:
# Fetch all items currently stored in the session
messages = await session.get_items()

# Print them out to inspect the data
for msg in messages:
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    print(f"[{role.upper()}]: {content}")

[USER]: Hi, I'm Aida.
[ASSISTANT]: [{'annotations': [], 'text': 'Hi Aida! How can I assist you today?', 'type': 'output_text', 'logprobs': []}]
[USER]: What's my name?
[ASSISTANT]: [{'annotations': [], 'text': 'Your name is Aida.', 'type': 'output_text', 'logprobs': []}]


In [23]:
# filtered inspection
recent_messages = await session.get_items(limit=4)
recent_messages

[{'content': "Hi, I'm Aida.", 'role': 'user'},
 {'id': 'msg_0ae7ec660dd6ad58006a35734d5f4c8192828832f3fafbae79',
  'content': [{'annotations': [],
    'text': 'Hi Aida! How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'},
 {'content': "What's my name?", 'role': 'user'},
 {'id': 'msg_0ae7ec660dd6ad58006a35734e41788192a00b6689fb954850',
  'content': [{'annotations': [],
    'text': 'Your name is Aida.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

## 2.4 Persistent sessions

🟢 **Core lesson**

So far the session has been in-memory — gone the moment Python exits.   
Pass a file path and the conversation survives a restart.

This still counts as **short-term memory** in our architecture. The session is scoped to one conversation thread. 
> *"Short-term"* describes its purpose (one conversation), not whether it's on disk or in RAM.

In [24]:
SESSION_DB = "conversations.db"
SESSION_ID = "persistent_demo"

# Open the session (creates the file if missing).
session_v1 = SQLiteSession(SESSION_ID, SESSION_DB)

await Runner.run(stateful, "My favourite colour is teal.", session=session_v1)

# A NEW Python object pointing at the same session ID and file.
# Simulates restarting the app and re-opening the conversation.
session_v2 = SQLiteSession(SESSION_ID, SESSION_DB)

result = await Runner.run(stateful, "What's my favourite colour?", session=session_v2)
print(f"Recovered after 'restart': {result.final_output}")

Recovered after 'restart': Your favorite color is teal!


### Session isolation

Different session IDs mean different conversation histories. They're completely independent.

In [25]:
session_aida = SQLiteSession("user_aida", SESSION_DB)
session_boris = SQLiteSession("user_boris", SESSION_DB)

await Runner.run(stateful, "I'm Aida. I love sci-fi books.", session=session_aida)
await Runner.run(stateful, "I'm Boris. I love history books.", session=session_boris)

aida = await Runner.run(stateful, "Recommend a book.", session=session_aida)
boris = await Runner.run(stateful, "Recommend a book.", session=session_boris)

print(f"Aida's recommendation:\n{aida.final_output}\n")
print(f"Boris's recommendation:\n{boris.final_output}")

Aida's recommendation:
I recommend *Dune* by Frank Herbert. It's a classic with a rich universe, political intrigue, and ecological themes. Have you read it?

Boris's recommendation:
I recommend *Sapiens: A Brief History of Humankind* by Yuval Noah Harari. It offers a compelling overview of human history, exploring how societies have evolved over time. It’s thought-provoking and accessible!


⚠️ **Production note — session ID naming.**

Always use a meaning session ID, that heps you organise the conversations.   
In a real application, derive session IDs from authenticated user identity, never from user input. Otherwise a malicious user could write *"my session ID is 'aida'"* and read Aida's history.

```python
SQLiteSession(f"user_{authenticated_user_id}")                          # one per user
SQLiteSession(f"user_{authenticated_user_id}_thread_{thread_id}")       # per-user, per-conversation
SQLiteSession(f"support_ticket_{ticket_id}")                            # tied to a business object
```

Pick one scheme, use it consistently, derive the ID from auth — not from user-controlled input.

## 2.5 Multi-turn agent with tools and memory

Putting Part 1 and Part 2 together — an agent that uses tools *and* remembers the conversation.

In [27]:
multi_turn_agent = Agent(
    name="Briefing Assistant",
    instructions=(
        "You are a helpful briefing assistant. "
        "Use get_crypto_price for crypto questions and get_weather for weather. "
        "Reference previous turns when relevant."
    ),
    tools=[get_crypto_price, get_weather],
    model=MODEL,
)

session = SQLiteSession("briefing_demo")

t1 = await Runner.run(
    multi_turn_agent,
    "Hi, I live in Almaty and I wanted to know what the price of bitcoin is in USD?",
    session=session,
)
print(f"Turn 1: {t1.final_output}\n")

t2 = await Runner.run(
    multi_turn_agent,
    "What about the weather where I live?",
    session=session,
)
print(f"Turn 2: {t2.final_output}")

Turn 1: The price of Bitcoin is currently $63,032 USD. If you have any other questions or need further information, feel free to ask!

Turn 2: The weather in Almaty is currently cold and clear, with a temperature of -3°C. If you need more information or assistance, just let me know!


Notice Turn 2 — the agent answered the weather question for Almaty *without* the user repeating the city. The session carried the context.

That's a real multi-turn agent.

## 2.6 Multiple agents sharing memory

A session isn't owned by one agent — it's a piece of shared state. Several agents can read and write to the same session, which lets specialists collaborate on the same conversation.

Below, three agents share one session:

- A **Researcher** that uses the SDK's built-in `WebSearchTool` to find current information
- An **Explainer** that takes the research and rewrites it in plain language
- An **Archivist** that saves the explanation to a markdown file

The session carries the conversation forward — each agent sees what the previous one did and builds on it.

In [28]:
from agents import WebSearchTool

researcher = Agent(
    name="Researcher",
    instructions=(
        "You research current topics. When the user asks about something recent, "
        "use WebSearchTool to find authoritative sources and summarise in 2-3 sentences. "
        "Cite the sources you used."
    ),
    tools=[WebSearchTool()],
    model=MODEL,
)

explainer = Agent(
    name="Explainer",
    instructions=(
        "You take research from the previous conversation turn and explain it in plain language, "
        "as if to a first-year student. Reference what was just researched. "
        "Keep it to a few short paragraphs."
    ),
    model=MODEL,
)

archivist = Agent(
    name="Archivist",
    instructions=(
        "You save the previous explanation to a file. "
        "Use the tool write_text_file with a sensible filename based on the topic discussed. "
        "Reply with one sentence confirming where it was saved."
    ),
    tools=[write_text_file],
    model=MODEL,
)

In [29]:
# One session shared by all three agents.
briefing_session = SQLiteSession("collab_demo")

# Turn 1 — Researcher fetches current information.
turn1 = await Runner.run(
    researcher,
    "What are the latest developments in small language models?",
    session=briefing_session,
)
print(f"🔎 Researcher:\n{turn1.final_output}\n")

# Turn 2 — Explainer rewrites in plain language, drawing on the research.
turn2 = await Runner.run(
    explainer,
    "Explain that to me like I'm a first-year student.",
    session=briefing_session,
)
print(f"📘 Explainer:\n{turn2.final_output}\n")

# Turn 3 — Archivist saves the explanation to a file.
turn3 = await Runner.run(
    archivist,
    "Save the explanation we just discussed to a file.",
    session=briefing_session,
)
print(f"💾 Archivist:\n{turn3.final_output}")

🔎 Researcher:
Recent advancements in small language models (SLMs) have significantly enhanced their performance and applicability across various domains. Notably, models under 3 billion parameters have achieved MMLU scores between 65% and 70%, approaching the capabilities of larger models like GPT-3.5, while maintaining efficiency suitable for on-device deployment. ([presenc.ai](https://presenc.ai/research/best-small-language-models-under-3b-2026?utm_source=openai))

Meta's MobileLLM-R1, a family of sub-billion parameter models, exemplifies this trend by delivering specialized reasoning capabilities for tasks such as math, coding, and scientific reasoning, all while operating on local devices. ([venturebeat.com](https://venturebeat.com/ai/metas-new-small-reasoning-model-shows-industry-shift-toward-tiny-ai-for?utm_source=openai))

Mistral's Small 4 model further consolidates reasoning, vision, and coding functionalities into a single, efficient model, reducing inference costs and latenc

**Look at what happened.** The Explainer answered as if it had done the research itself — because the session contained the Researcher's earlier turn. Same for the Archivist: it saved the Explainer's plain-language version without anyone passing it explicitly. **The session is the common ground.**

Open the saved file in `outputs/` to see the result. This is genuinely useful — three specialists, three different jobs, one coherent conversation that produces a real artefact.

This pattern of *sharing session state between specialists* is the foundation for the multi-agent systems you'll meet on **Day 3**. There, we'll go deeper — when to use which orchestration pattern, how to make handoffs reliable, how to debug what each agent did.

⚠️ **Production note:** sharing a session works, but each agent still has its own instructions and tools. When agents conflict — one writes facts, another contradicts them — behaviour can become unpredictable. Multi-agent design is its own engineering skill.

## 2.7 Chat Loop (with Memory and Tools) [OPTIONAL EXERCISE]

### Non Persistent (In-Memory)

In [30]:
# Create the agent and session outside the function so they live in the notebook memory
agent = Agent(
    name="Assistant", instructions="You are a helpful, concise assistant.",
    tools=[get_crypto_price, get_weather],
    model=MODEL)

non_persisten_session = SQLiteSession("non_persisten_chat")

print("🤖 Chat started (In-Memory)! Type 'quit' or 'exit' to stop.")

# Run the loop directly in the cell
while True:
    user_input = input("\nYou: ").strip()
    
    if user_input.lower() in ["quit", "exit"]:
        print("Goodbye!")
        break
        
    if not user_input:
        continue
        
    # We can use direct top-level 'await' inside Jupyter cells!
    result = await Runner.run(agent, user_input, session=non_persisten_session)
    print(f"Agent: {result.final_output}")


🤖 Chat started (In-Memory)! Type 'quit' or 'exit' to stop.
Agent: Nice to meet you, Sam! How can I assist you today?
Agent: It seems I couldn't retrieve the weather data for Tokyo. Would you like to check a different city or try something else?
Agent: The current price of Bitcoin (BTC) is £47,556. Is there anything else you need?
Goodbye!


### Persistent (Database File)

In [32]:
# Create the agent outside the loop
agent = Agent(
    name="Assistant", 
    instructions="You are a helpful, concise assistant.",
    tools=[get_crypto_price, get_weather],
    model=MODEL
)

# 🟢 CHANGE: Use a clear connection string pointing to a real local database file
# This ensures it saves to your hard drive instead of RAM
persistent_session = SQLiteSession("persistent_chat",db_path="chat_history.db")

print("💾 Chat started (Persistent Database File)! Type 'quit' or 'exit' to stop.")

# Run the loop directly in the cell
while True:
    user_input = input("\nYou: ").strip()
    
    if user_input.lower() in ["quit", "exit"]:
        print("Goodbye!")
        break
        
    if not user_input:
        continue
        
    # Run using your persistent session
    result = await Runner.run(agent, user_input, session=persistent_session)
    print(f"Agent: {result.final_output}")


💾 Chat started (Persistent Database File)! Type 'quit' or 'exit' to stop.
Agent: Your lucky number is 15!
Goodbye!


---

# Part 3 — Making it production-ready

Notebooks are great for learning. They're terrible for production.

**The difference between a notebook that works and code you'd ship comes down to four things:**

1. **Testing.** You can verify the code does what you expect, repeatedly, without running it manually.
2. **Error handling.** When something goes wrong, the system fails *gracefully* — clear messages, useful logs, no silent corruption.
3. **Observability.** You can see what your code is doing in production, not just at development time.
4. **Structure.** The code is organised so other engineers can read, modify, and reuse it.

This part covers the first three. The fourth — structure — is something you'll see at the very end of the day, when we look at how the notebook code can be re-organised into the `src/agent_workshop/` modules that ship as part of this course's repo.

## 3.1 What "production-ready" means for agent code

🟢 **Core lesson**

A working notebook is one where the cells run top-to-bottom without error. That bar is *much* lower than production.

Production-ready agent code:

- **Handles tool failures gracefully** — agents recover from a bad API call instead of crashing
- **Has tests** — you can change something without holding your breath
- **Logs what it does** — when behaviour is weird, you can see why
- **Has clear boundaries** — tools, prompts, and agents live in separate, testable units

These aren't optional in industry. They're table stakes.

## 3.2 Pytest patterns for tools

🟢 **Core lesson**

Tools are normal Python functions, which means they can be unit-tested. The catch: a `@function_tool`-decorated function is wrapped — you can't call it directly with arguments in the same way.

**The pattern that solves this:** write the tool as *two pieces*. A plain Python function (the logic), and a `@function_tool` wrapper. Tests use the plain function. Agents use the wrapper.

This is the same pattern you'll see in the course's `course_tools` module — already on disk in the repo.

In [33]:
# The two-step pattern.

def _greet(name: str, formal: bool = False) -> str:
    """The actual logic — testable as a normal Python function."""
    if not name.strip():
        return "Error: name cannot be empty."
    if formal:
        return f"Good day, {name}."
    return f"Hi {name}!"


# Wrap it as a tool for agents to use.
greet = function_tool(_greet)


# Now you can test the underscore version directly.
assert _greet("Aida") == "Hi Aida!"
assert _greet("Aida", formal=True) == "Good day, Aida."
assert _greet("").startswith("Error:")

print("All assertions passed.")

All assertions passed.


### What a real pytest file looks like

Here's how `_greet` would be tested in a proper test file (e.g. `tests/test_greet.py`):

```python
# tests/test_greet.py
from agent_workshop.tools.greet import _greet


def test_casual_greeting():
    assert _greet("Aida") == "Hi Aida!"


def test_formal_greeting():
    assert _greet("Aida", formal=True) == "Good day, Aida."


def test_empty_name_returns_error():
    result = _greet("")
    assert result.startswith("Error:")


def test_whitespace_name_rejected():
    result = _greet("   ")
    assert result.startswith("Error:")
```

Run with:

```bash
poetry run pytest tests/ -v
```

✅ **Check your understanding:** write three pytest test cases for the `_write_text_file` function (the underscore version of `write_text_file` from Section 1.5). Test the happy path, the path-separator rejection, and what happens when the directory doesn't exist.

### Testing tools that call external APIs

External APIs make tests slow and unreliable. The fix: **mock the network call**, so tests run offline and deterministically.

```python
# tests/test_crypto.py
from pytest_mock import MockerFixture

from agent_workshop.tools.crypto import _get_crypto_price


def test_get_crypto_price_success(mocker: MockerFixture):
    fake_response = mocker.Mock()
    fake_response.json.return_value = {"bitcoin": {"usd": 50000}}
    fake_response.raise_for_status.return_value = None

    mocker.patch("agent_workshop.tools.crypto.httpx.get", return_value=fake_response)

    result = _get_crypto_price("bitcoin", "usd")
    assert "50000" in result
```

This test runs in milliseconds and never touches the real CoinGecko API. It catches regressions in your *code*, not theirs.

Run with:

```bash
poetry run pytest tests/test_crypto.py -v
```

## 3.3 Writing error messages that help the agent

🟢 **Core lesson**

You've already seen Part 1's tools return errors as strings instead of raising exceptions. That's the first principle. The second — and more important — principle is **how those error messages are written**.

When a tool returns an error, the agent reads it like any other tool result. It then has to decide what to do next: retry, ask the user, try a different tool, give up. **The quality of that decision depends entirely on the error message.**

Error messages are part of your tool's API. Write them for the agent, not for a log file.



### Useless errors vs useful errors

Compare these two implementations of the same failure:

In [34]:
# ❌ Useless — the agent has nothing to act on
def _bad_lookup(symbol: str) -> str:
    if symbol not in {"bitcoin", "ethereum"}:
        return "Error"
    return f"Price: 50000"


# ❌ Slightly better, but still bad
def _meh_lookup(symbol: str) -> str:
    if symbol not in {"bitcoin", "ethereum"}:
        return "Symbol not found"
    return f"Price: 50000"


# ✅ Good — actionable, specific, suggests next steps
def _good_lookup(symbol: str) -> str:
    valid = {"bitcoin", "ethereum"}
    if symbol not in valid:
        return (
            f"Error: '{symbol}' is not a recognised symbol. "
            f"Try one of: {', '.join(sorted(valid))}."
        )
    return f"Price: 50000"


print(_bad_lookup("btc"))    # Error
print(_meh_lookup("btc"))    # Symbol not found
print(_good_lookup("btc"))   # Error: 'btc' is not recognised. Try one of: bitcoin, ethereum.

Error
Symbol not found
Error: 'btc' is not a recognised symbol. Try one of: bitcoin, ethereum.


### Four principles for actionable error messages

When you write an error message for an agent, ask these four questions:

| Principle | Bad | Good |
|---|---|---|
| **Be specific about what failed** | `"Error"` | `"Error: 'btc' is not a recognised symbol."` |
| **Suggest a next step** | `"Symbol not found"` | `"Try 'bitcoin', 'ethereum', or 'solana'."` |
| **Give the agent useful context** | `"File not found"` | `"File 'notes.md' not found. Available files: report.md, draft.md."` |
| **Distinguish failure modes** | `"API error"` | `"Rate limit hit — wait 30 seconds before retrying."` vs `"CoinGecko unreachable — check connection."` |

### Different failures need different messages

The classic mistake is catching every exception with one generic message. The agent then can't tell *what* went wrong, so it can't react differently.

In [35]:
# ❌ Every failure looks the same
def _bad_fetch(symbol: str) -> str:
    try:
        response = httpx.get(URL, timeout=10.0)
        response.raise_for_status()
        return parse(response.json(), symbol)
    except Exception:
        return "Error: could not fetch price."


# ✅ Each failure mode tells the agent something different
def _good_fetch(symbol: str) -> str:
    try:
        response = httpx.get(URL, timeout=10.0)
        response.raise_for_status()
    except httpx.TimeoutException:
        return "Error: request timed out. Try again in a few seconds."
    except httpx.HTTPStatusError as exc:
        if exc.response.status_code == 429:
            return "Error: rate limit hit. Wait 30 seconds before retrying."
        return f"Error: HTTP {exc.response.status_code} from the price API."

    return parse(response.json(), symbol)

A timeout means *retry*. A rate limit means *wait*. A 500 means *try a different tool*. **The agent can only make those distinctions if your error messages do.**

## 3.4 Observability — what to log and trace

🟢 **Core lesson**

When something goes wrong in production, you won't be there to see it happen. **Logs and traces are how you find out what happened later.**

For an agent system, three things are worth recording:

| Signal | What to capture | Why |
|---|---|---|
| **Tool calls** | name, arguments, result, duration | Most agent issues are tool issues |
| **Token usage** | input + output tokens per turn | Cost adds up fast |
| **Errors** | exception types, error messages, stack traces | Debug what failed |

The Agents SDK includes a built-in tracer that captures all three for free. We'll go deeper into tracing on Day 3 (when traces become essential for multi-agent debugging) — but it's worth knowing the SDK supports it from day one.

### The simplest useful logging

For your own code (not tool logic), `logging` from the standard library is enough:

```python
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def _get_crypto_price(symbol: str) -> str:
    logger.info(f"Fetching price for {symbol}")
    try:
        # ...
    except Exception as exc:
        logger.error(f"Crypto price fetch failed for {symbol}: {exc}")
        return f"Error: {exc}"
```

In production, those `logger.info` and `logger.error` calls become searchable records. You can grep them. You can alert on them. You can graph them.

📚 **For the future:** production observability often means **Langfuse**, **LangSmith**, or **OpenTelemetry** — tools that capture all of this for you and give you dashboards. Out of scope for today, but worth knowing they exist.

## 3.5 From notebook to source modules

The notebook has built every tool, agent, and pattern directly in cells. That's good for learning. It's bad for shipping.

The same code is also available in this course's repo as proper Python modules:

```text
src/agent_workshop/
├── __init__.py            # public API
├── config.py              # Pydantic Settings, .env loading
├── prompts.py             # prompt templates
├── tools/
│   ├── __init__.py
│   ├── weather.py
│   ├── crypto.py
│   ├── search.py
│   ├── files.py
│   └── prices.py
└── agents.py              # build_assistant, build_briefer
```

And as CLI entry points:

```bash
poetry run python scripts/chat.py
poetry run python scripts/ask.py "what's the price of bitcoin?"
```

These are a *refactoring* of what you saw in the notebook, not a separate design. The patterns are the same — only the structure changes. Open them after the lab and read through. **This is what the production-ready version of today's work looks like.**

Tests live in `tests/`, run with `poetry run pytest tests/ -v`.

---

## Recap

Six things to take into the lab:

1. **A tool is just a Python function** with a name, description, and signature the model can read.
2. **Typed inputs make tools predictable.** Use `Literal[...]` for known options and Pydantic for structured inputs.
3. **Structured outputs make agent responses usable by other code.** Pass `output_type=` to `Agent(...)`.
4. **Sessions give the agent multi-turn memory.** Pass the same session to multiple `Runner.run()` calls.
5. **Tools should return errors, not raise them.** The agent can recover from a string. It can't recover from a crash.
6. **Production-ready code is tested, observable, and structured.** Today's tools and patterns are designed to be all three.

## Afternoon lab brief

Build a multi-turn agent that **does something useful with at least two tools**.

Requirements:

- Use at least **two tools** — pick from today's notebook (web search, crypto, gold, file I/O) or write your own
- Use **structured output** for the agent's response, where it makes sense
- Use a **session** so the agent remembers the conversation across at least three turns
- Handle at least **one error case** gracefully (a missing argument, a failed API call, etc.)
- Demo your agent at the end of the lab

Some ideas:

- A **personal briefing agent** — weather + crypto + news, remembers your city and preferred coins
- A **research assistant** — web search + write to file, remembers what you've already researched
- A **commodity tracker** — gold + crypto + currency converter, remembers your target prices

If you finish early, write at least one pytest test for a tool you used.

This is also the system you'll extend on Day 3 (multi-agent) and possibly Day 5 (capstone). Build something you'd like to use.

See you in the lab.